In [6]:
# ── LSTM.ipynb — imports, reproducibility, device ────────────────────────
import numpy as np
import pandas as pd
import torch
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error

import features as F

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print("device:", device)          # 'mps' on Apple Silicon — much faster than cpu

SEQ_LEN  = 55
VAL_FRAC = 0.2

device: mps


In [7]:
# ── Load + engineer features (shared module) ─────────────────────────────
df = pd.read_csv("Data/train.csv")
df = F.build_features(df)                 # sorted (stock, date, seconds) + features
feat_cols = F.feature_columns(df)
print(f"{len(feat_cols)} per-timestep features")

33 per-timestep features


In [8]:
# ── Scale (fit on TRAIN dates only) + clean NaN/inf ──────────────────────
dates  = np.sort(df["date_id"].unique())
cutoff = dates[-int(len(dates) * VAL_FRAC)]        # first validation date

X_all    = df[feat_cols].replace([np.inf, -np.inf], np.nan).fillna(0.0)
scaler   = StandardScaler().fit(X_all[df["date_id"] < cutoff])
X_scaled = scaler.transform(X_all).astype(np.float32)

In [9]:
# ── Reshape flat rows -> (n_seq, 55, n_feat) sequences ──────────────────
counts = df.groupby(["stock_id", "date_id"]).size()
assert (counts == SEQ_LEN).all(), "a stock-day isn't 55 steps long"
n_seq = len(counts)

X_seq    = X_scaled.reshape(n_seq, SEQ_LEN, len(feat_cols))
y_seq    = df["target"].to_numpy(np.float32).reshape(n_seq, SEQ_LEN)
mask_seq = (~np.isnan(y_seq)).astype(np.float32)     # 1=valid target, 0=NaN
y_seq    = np.nan_to_num(y_seq, nan=0.0)

first_rows = df.iloc[::SEQ_LEN]                       # first step of each stock-day
seq_date   = first_rows["date_id"].to_numpy()
stock_ids  = np.sort(df["stock_id"].unique())
stock_to_idx  = {s: i for i, s in enumerate(stock_ids)}
seq_stock_idx = first_rows["stock_id"].map(stock_to_idx).to_numpy(np.int64)

In [10]:
# ── Split sequences by date, wrap in tensors + loaders ──────────────────
tr = seq_date < cutoff
def to_tensors(idx):
    return (torch.tensor(X_seq[idx]),       torch.tensor(seq_stock_idx[idx]),
            torch.tensor(y_seq[idx]),       torch.tensor(mask_seq[idx]))

Xtr, Str, ytr, mtr = to_tensors(tr)
Xva, Sva, yva, mva = to_tensors(~tr)

train_loader = DataLoader(TensorDataset(Xtr, Str, ytr, mtr), batch_size=256, shuffle=True)
val_loader   = DataLoader(TensorDataset(Xva, Sva, yva, mva), batch_size=512, shuffle=False)
n_features, n_stocks = len(feat_cols), len(stock_ids)
print(f"train {tr.sum():,} | val {(~tr).sum():,} seqs | X {tuple(Xtr.shape)} | "
      f"{n_features} feats, {n_stocks} stocks")

train 76,036 | val 19,200 seqs | X (76036, 55, 33) | 33 feats, 200 stocks


## Base Model

In [11]:
# ── Model: stock embedding + unidirectional RNN -> per-timestep output ────
import torch.nn as nn

class SeqModel(nn.Module):
    """GRU or LSTM over the 55-step auction sequence. Predicts a target at
    EVERY timestep. Unidirectional by design — see the note below."""
    def __init__(self, n_features, n_stocks, rnn="gru",
                 emb_dim=24, hidden=128, layers=2, dropout=0.2):
        super().__init__()
        self.emb = nn.Embedding(n_stocks, emb_dim)
        rnn_cls = nn.GRU if rnn == "gru" else nn.LSTM
        self.rnn = rnn_cls(
            input_size=n_features + emb_dim,
            hidden_size=hidden, num_layers=layers,
            batch_first=True, dropout=dropout,
            bidirectional=False,        # <-- CRITICAL: never True (see note)
        )
        self.head = nn.Sequential(nn.Linear(hidden, 64), nn.ReLU(), nn.Linear(64, 1))

    def forward(self, x, stock):                     # x: (B,55,F)  stock: (B,)
        e = self.emb(stock).unsqueeze(1).expand(-1, x.size(1), -1)  # (B,55,emb)
        out, _ = self.rnn(torch.cat([x, e], dim=-1))               # (B,55,hidden)
        return self.head(out).squeeze(-1)                          # (B,55)

In [12]:
# ── Masked MAE: the 88 NaN-target steps must contribute zero loss ────────
def masked_mae(pred, y, mask):
    return (torch.abs(pred - y) * mask).sum() / mask.sum()

@torch.no_grad()
def eval_mae(model, loader):
    """Exact masked MAE over a loader (matches the competition metric)."""
    model.eval()
    err, n = 0.0, 0.0
    for x, s, y, m in loader:
        x, s, y, m = x.to(device), s.to(device), y.to(device), m.to(device)
        p = model(x, s)
        err += (torch.abs(p - y) * m).sum().item()
        n   += m.sum().item()
    return err / n

In [8]:
# ── Train with early stopping on validation MAE ──────────────────────────
model = SeqModel(n_features, n_stocks, rnn="gru").to(device)
opt   = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)

val_zero = eval_mae(  # zero-prediction MAE on val, for skill % (uses a zero model)
    type("Z", (), {"eval": lambda s: None, "__call__": lambda s, x, st: torch.zeros_like(x[..., 0])})(),
    val_loader) if False else float((torch.abs(yva) * mva).sum() / mva.sum())

best, best_state, patience, bad = 1e9, None, 4, 0
for epoch in range(40):
    model.train()
    for x, s, y, m in train_loader:
        x, s, y, m = x.to(device), s.to(device), y.to(device), m.to(device)
        opt.zero_grad()
        loss = masked_mae(model(x, s), y, m)
        loss.backward()
        opt.step()

    v = eval_mae(model, val_loader)
    skill = (val_zero - v) / val_zero * 100
    print(f"epoch {epoch:2d}  val MAE {v:.4f}  skill {skill:.2f}%")

    if v < best - 1e-4:
        best, best_state, bad = v, {k: t.cpu().clone() for k, t in model.state_dict().items()}, 0
    else:
        bad += 1
        if bad >= patience:
            print(f"early stop — best val MAE {best:.4f}"); break

model.load_state_dict(best_state)
print(f"BEST val MAE {best:.4f}  (LightGBM was ~5.9375)")

epoch  0  val MAE 5.9752  skill 1.40%
epoch  1  val MAE 5.9636  skill 1.59%
epoch  2  val MAE 5.9699  skill 1.49%
epoch  3  val MAE 5.9551  skill 1.73%
epoch  4  val MAE 5.9567  skill 1.71%
epoch  5  val MAE 5.9460  skill 1.88%
epoch  6  val MAE 5.9501  skill 1.81%
epoch  7  val MAE 5.9450  skill 1.90%
epoch  8  val MAE 5.9473  skill 1.86%
epoch  9  val MAE 5.9444  skill 1.91%
epoch 10  val MAE 5.9501  skill 1.81%
epoch 11  val MAE 5.9475  skill 1.86%
epoch 12  val MAE 5.9482  skill 1.85%
epoch 13  val MAE 5.9577  skill 1.69%
early stop — best val MAE 5.9444
BEST val MAE 5.9444  (LightGBM was ~5.9375)


In [13]:
# GRU predictions in the SAME flat order as df (sequences flatten row-for-row)
@torch.no_grad()
def predict_flat(model, X_seq, stock_idx, batch=512):
    model.eval()
    out = []
    for i in range(0, len(X_seq), batch):
        x = torch.tensor(X_seq[i:i+batch]).to(device)
        s = torch.tensor(stock_idx[i:i+batch]).to(device)
        out.append(model(x, s).cpu().numpy())          # (b, 55)
    return np.concatenate(out).reshape(-1)              # aligned to df rows

df_out = df[["row_id", "stock_id", "date_id", "seconds_in_bucket", "target"]].copy()
df_out["gru"] = predict_flat(model, X_seq, seq_stock_idx)
df_out[df_out.date_id >= cutoff].to_parquet("preds_gru_val.parquet", index=False)
print("saved GRU val preds")

NameError: name 'model' is not defined

In [10]:
# ── Multi-seed GRU: train N seeds, average their val predictions ─────────
def train_one(seed):
    torch.manual_seed(seed); np.random.seed(seed)
    model = SeqModel(n_features, n_stocks, rnn="gru").to(device)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
    best, best_state, bad = 1e9, None, 0
    for epoch in range(40):
        model.train()
        for x, s, y, mk in train_loader:
            x, s, y, mk = x.to(device), s.to(device), y.to(device), mk.to(device)
            opt.zero_grad(); masked_mae(model(x, s), y, mk).backward(); opt.step()
        v = eval_mae(model, val_loader)
        if v < best - 1e-4: best, best_state, bad = v, {k: t.cpu().clone() for k, t in model.state_dict().items()}, 0
        else:
            bad += 1
            if bad >= 4: break
    model.load_state_dict(best_state)
    print(f"  seed {seed}: val MAE {best:.4f}")
    return predict_flat(model, X_seq, seq_stock_idx)      # flat preds aligned to df

seeds = [0, 1, 2]
gru_multi = np.mean([train_one(s) for s in seeds], axis=0)   # averaged predictions

df_out = df[["row_id", "date_id", "seconds_in_bucket", "target"]].copy()
df_out["gru"] = gru_multi
df_out[df_out.date_id >= cutoff].to_parquet("preds_gru_val.parquet", index=False)
print("saved multi-seed GRU preds — now re-run the blend cell")

  seed 0: val MAE 5.9425
  seed 1: val MAE 5.9470
  seed 2: val MAE 5.9462
saved multi-seed GRU preds — now re-run the blend cell


In [14]:
import json, numpy as np, os
os.makedirs("artifacts", exist_ok=True)
torch.save(model.state_dict(), "artifacts/gru.pt")
np.savez("artifacts/scaler.npz", mean=scaler.mean_, scale=scaler.scale_)
json.dump({
    "feat_cols": feat_cols,
    "stock_to_idx": {int(k): int(v) for k, v in stock_to_idx.items()},
    "n_features": len(feat_cols), "n_stocks": len(stock_to_idx), "blend_w_lgb": 0.5,
}, open("artifacts/meta.json", "w"))
print("GRU artifacts saved")

NameError: name 'model' is not defined

In [12]:
import gc, lightgbm as lgb
# release everything the LGB step doesn't need
for v in ["X_all", "X_scaled", "X_seq", "y_seq", "mask_seq",
          "Xtr", "Xva", "ytr", "yva", "mtr", "mva", "train_loader", "val_loader"]:
    if v in globals(): del globals()[v]
gc.collect()

cols = feat_cols + ["stock_id"]
Xf = df[cols].replace([np.inf, -np.inf], np.nan).fillna(0)   # reuse df — no rebuild
yf = df["target"]; ok = yf.notna().values
gbm = lgb.LGBMRegressor(objective="mae", n_estimators=500, learning_rate=0.03,
    num_leaves=31, min_child_samples=1000, reg_alpha=1.0, reg_lambda=1.0,
    subsample=0.7, subsample_freq=1, colsample_bytree=0.7, random_state=42,
    n_jobs=-1, verbose=-1).fit(Xf[ok], yf[ok], categorical_feature=["stock_id"])
gbm.booster_.save_model("artifacts/lgb_final.txt")
print("LGB saved — artifacts/:", os.listdir("artifacts"))

: 

## New Model

In [19]:
df = pd.read_csv("Data/train.csv")
df = F.build_features(df)
#df = F.add_global_features(df)      # +5 market-wide features the RNN can't derive
feat_cols = F.feature_columns(df)   # now 38 instead of 33
print(f"{len(feat_cols)} per-timestep features")

33 per-timestep features


In [16]:
def train_one(seed, rnn="gru"):
    torch.manual_seed(seed); np.random.seed(seed)
    model = SeqModel(n_features, n_stocks, rnn=rnn).to(device)
    opt   = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", factor=0.5, patience=2)
    best, best_state, patience, bad = 1e9, None, 7, 0      # patience 4 -> 7

    for epoch in range(50):
        model.train()
        for x, s, y, m in train_loader:
            x, s, y, m = x.to(device), s.to(device), y.to(device), m.to(device)
            opt.zero_grad()
            loss = masked_mae(model(x, s), y, m)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)   # <-- grad clip
            opt.step()

        v = eval_mae(model, val_loader)
        sched.step(v)                                                 # <-- LR decay on plateau
        if v < best - 1e-4:
            best, best_state, bad = v, {k: t.cpu().clone() for k, t in model.state_dict().items()}, 0
        else:
            bad += 1
            if bad >= patience: break
    model.load_state_dict(best_state)
    print(f"  seed {seed} ({rnn}): val MAE {best:.4f}")
    return predict_flat(model, X_seq, seq_stock_idx)

In [17]:
seeds = [0, 1, 2]
gru_multi = np.mean([train_one(s) for s in seeds], axis=0)   # averaged predictions

df_out = df[["row_id", "date_id", "seconds_in_bucket", "target"]].copy()
df_out["gru"] = gru_multi
df_out[df_out.date_id >= cutoff].to_parquet("preds_gru_val.parquet", index=False)
print("saved multi-seed GRU preds — now re-run the blend cell")

  seed 0 (gru): val MAE 5.9421
  seed 1 (gru): val MAE 5.9471
  seed 2 (gru): val MAE 5.9440
saved multi-seed GRU preds — now re-run the blend cell


In [18]:
seeds = [0, 1, 2]
lstm_multi = np.mean([train_one(s, rnn="lstm") for s in seeds], axis=0)   # averaged predictions

df_out = df[["row_id", "date_id", "seconds_in_bucket", "target"]].copy()
df_out["lstm"] = lstm_multi
df_out[df_out.date_id >= cutoff].to_parquet("preds_lstm_val.parquet", index=False)
print("saved multi-seed LSTM preds — now re-run the blend cell")

  seed 0 (lstm): val MAE 5.9529
  seed 1 (lstm): val MAE 5.9524
  seed 2 (lstm): val MAE 5.9522
saved multi-seed LSTM preds — now re-run the blend cell


In [20]:
def train_and_save(seed, epochs=40):
    torch.manual_seed(seed); np.random.seed(seed)
    model = SeqModel(n_features, n_stocks, rnn="gru").to(device)
    opt   = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", factor=0.5, patience=2)
    best, best_state, patience, bad = 1e9, None, 7, 0
    for ep in range(epochs):
        model.train()
        for x, s, y, m in train_loader:
            x, s, y, m = x.to(device), s.to(device), y.to(device), m.to(device)
            opt.zero_grad(); masked_mae(model(x, s), y, m).backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
        v = eval_mae(model, val_loader); sched.step(v)
        if v < best - 1e-4:
            best, best_state, bad = v, {k: t.cpu().clone() for k, t in model.state_dict().items()}, 0
        else:
            bad += 1
            if bad >= patience: break
    model.load_state_dict(best_state)
    torch.save(best_state, f"artifacts/gru_seed{seed}.pt")
    print(f"seed {seed}: val MAE {best:.4f}")
    return predict_flat(model, X_seq, seq_stock_idx)

gru_multi = np.mean([train_and_save(s) for s in [0, 1, 2]], axis=0)
df_out = df[["row_id","date_id","seconds_in_bucket","target"]].copy()
df_out["gru"] = gru_multi
df_out[df_out.date_id >= cutoff].to_parquet("preds_gru_val.parquet", index=False)
print("saved artifacts/gru_seed*.pt + preds_gru_val.parquet (base 33)")

seed 0: val MAE 5.9424
seed 1: val MAE 5.9471
seed 2: val MAE 5.9448
saved artifacts/gru_seed*.pt + preds_gru_val.parquet (base 33)
